In [1]:

!nvidia-smi

# Install required packages
!pip install -q transformers datasets torch mlflow==3.1.4 boto3 scikit-learn google-cloud-storage dvc[gs]

Tue Jan 27 15:03:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!unzip processed.zip

Archive:  processed.zip
   creating: processed/
  inflating: processed/dataset_dict.json  
   creating: processed/train/
  inflating: processed/train/data-00000-of-00001.arrow  
  inflating: processed/train/state.json  
  inflating: processed/train/dataset_info.json  
   creating: processed/validation/
  inflating: processed/validation/data-00000-of-00001.arrow  
  inflating: processed/validation/state.json  
  inflating: processed/validation/dataset_info.json  
   creating: processed/test/
  inflating: processed/test/data-00000-of-00001.arrow  
  inflating: processed/test/state.json  
  inflating: processed/test/dataset_info.json  
  inflating: processed/label_config.json  


In [3]:
from datasets import load_from_disk
import json
from transformers import (
  AutoModelForSequenceClassification,
  AutoTokenizer,
  Trainer,
  TrainingArguments,
  pipeline,
)
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
dataset = load_from_disk("/content/processed")
with open('/content/processed/label_config.json', 'r') as f:
    label_config = json.load(f)
print(f"\n🏷️  Number of labels: {label_config['num_labels']}")
print(f"   Entity types: {len(label_config['label_names'])} labels")



🏷️  Number of labels: 21
   Entity types: 21 labels


In [27]:
import mlflow

In [28]:
MLFLOW_TRACKING_URI = "http://34.126.82.240:5000/"

In [29]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [30]:
mlflow.set_experiment("ner")

2026/01/27 15:34:00 INFO mlflow.tracking.fluent: Experiment with name 'ner' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow/artifacts/1', creation_time=1769528040733, experiment_id='1', last_update_time=1769528040733, lifecycle_stage='active', name='ner', tags={}>

In [ ]:
client = mlflow.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
experiments = client.search_experiments()
print(f"\n✅ MLflow connection successful!")
print(f"   Total experiments: {len(experiments)}")



✅ MLflow connection successful!
   Total experiments: 3


In [ ]:
experiment = mlflow.get_experiment_by_name("ner")
experiment_id = experiment.experiment_id
print(f"name :{experiment.name}")
print(f"id :{experiment_id}")

name :ner
id :3


In [31]:
sample = dataset['train'][0]
print(f"   Input IDs length: {len(sample['input_ids'])}")
print(f"   Labels length: {len(sample['labels'])}")
print(f"   First 10 input IDs: {sample['input_ids'][:10]}")
print(f"   First 10 labels: {sample['labels'][:10]}")

   Input IDs length: 200
   Labels length: 200
   First 10 input IDs: [0, 2316, 790, 4, 326, 2142, 917, 9170, 2927, 380]
   First 10 labels: [-100, 20, 20, 20, 20, 20, 20, 20, 20, 20]


In [32]:
TRAINING_CONFIG = {
    'model_name': 'vinai/phobert-base-v2',
    'num_labels': label_config['num_labels'],
    'epochs': 4,
    'batch_size': 16,
    'learning_rate': 5e-5,
    'weight_decay': 0.01,
    'warmup_steps': 500,
    'max_grad_norm': 1.0,
    'logging_steps': 100,
    'eval_steps': 500,
    'save_steps': 500,
    'output_dir': 'models/phobert-ner',
}

In [33]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
)


In [34]:
tokenizer = AutoTokenizer.from_pretrained(TRAINING_CONFIG['model_name'], use_fast=True)

In [35]:
id2label = {int(k): v for k, v in label_config['id_to_label'].items()}

In [36]:
model = AutoModelForTokenClassification.from_pretrained(
    TRAINING_CONFIG['model_name'],
    num_labels=TRAINING_CONFIG['num_labels'],
    id2label=id2label,
    label2id=label_config['label_to_id'],
)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [37]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
            

In [38]:
from sklearn.metrics import classification_report, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    """
    Compute metrics for NER evaluation
    Ignores -100 labels (padding/special tokens)
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_labels = []
    true_predictions = []

    for prediction, label in zip(predictions, labels):
        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:  # Ignore padding
                true_labels.append(label_id)
                true_predictions.append(pred_id)

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average='weighted',
        zero_division=0
    )

    # Calculate per-class metrics
    precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        average=None,
        zero_division=0
    )

    # Calculate accuracy
    accuracy = np.mean(np.array(true_predictions) == np.array(true_labels))

    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

    # Add per-class F1 for important classes (entities)
    id_to_label = label_config['id_to_label']
    for idx, (p, r, f) in enumerate(zip(precision_per_class, recall_per_class, f1_per_class)):
        label = id_to_label.get(str(idx), f"label_{idx}")
        if label != 'O' and label.startswith('B-'):  # Only B- tags
            entity_type = label.split('-')[1]
            metrics[f'f1_{entity_type}'] = f

    return metrics

In [39]:
from transformers import TrainingArguments, DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
print("✅ Data collator created")

# Training arguments
training_args = TrainingArguments(
    output_dir=TRAINING_CONFIG['output_dir'],

    # Training schedule
    num_train_epochs=TRAINING_CONFIG['epochs'],
    per_device_train_batch_size=TRAINING_CONFIG['batch_size'],
    per_device_eval_batch_size=TRAINING_CONFIG['batch_size'],

    # Optimization
    learning_rate=TRAINING_CONFIG['learning_rate'],
    weight_decay=TRAINING_CONFIG['weight_decay'],
    warmup_steps=TRAINING_CONFIG['warmup_steps'],
    max_grad_norm=TRAINING_CONFIG['max_grad_norm'],

    # Evaluation and logging
    eval_strategy='steps',
    eval_steps=TRAINING_CONFIG['eval_steps'],
    logging_steps=TRAINING_CONFIG['logging_steps'],
    logging_dir=f"{TRAINING_CONFIG['output_dir']}/logs",

    # Saving
    save_strategy='steps',
    save_steps=TRAINING_CONFIG['save_steps'],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,

    # Performance
    fp16=True,  # Mixed precision training on GPU
    dataloader_num_workers=2,

    # Misc
    report_to='mlflow',  # We use MLflow for logging
    push_to_hub=False,
    disable_tqdm=False,
)


✅ Data collator created


In [40]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-93209024.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [41]:
mlflow.set_experiment("ner")

<Experiment: artifact_location='s3://mlflow/artifacts/1', creation_time=1769528040733, experiment_id='1', last_update_time=1769528040733, lifecycle_stage='active', name='ner', tags={}>

In [42]:

with mlflow.start_run() as run:
    trainer.train()


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,F1 Age,F1 Date,F1 Gender,F1 Job,F1 Location,F1 Name,F1 Organization,F1 Patient Id,F1 Symptom And Disease,F1 Transportation
500,0.170700,0.168666,0.971547,0.967016,0.971547,0.969168,0.926509,0.985182,0.914591,0.000000,0.945395,0.940217,0.903461,0.976636,0.888889,0.988372
1000,0.060000,0.083100,0.981572,0.981384,0.981572,0.981439,0.983333,0.988724,0.966372,0.798479,0.973448,0.946809,0.931003,0.988254,0.909332,1.000000


🏃 View run whimsical-sheep-667 at: http://34.126.82.240:5000/#/experiments/1/runs/562ac232d754457996a75a0ce42358a6
🧪 View experiment at: http://34.126.82.240:5000/#/experiments/1


In [43]:
import os
import torch
# import pipeline

In [49]:
os.environ['MLFLOW_S3_ENDPOINT_URL'] = "http://34.87.146.58:9000" # Cổng MinIO bạn đã sửa ở bước trước
os.environ['AWS_ACCESS_KEY_ID'] = "minioadmin"                # Mặc định của MinIO
os.environ['AWS_SECRET_ACCESS_KEY'] = "minioadmin123"

In [50]:
tuned_pipeline = pipeline(
  task="ner",
  model=trainer.model,
  batch_size=8,
  tokenizer=tokenizer,
  device="cuda",
)

Device set to use cuda


In [51]:
quick_check = (
    "Bệnh nhân Nguyễn Văn A, 35 tuổi, trú tại Cầu Giấy, Hà Nội. "
    "Nhập viện ngày 20/01/2026 với triệu chứng sốt cao và ho kéo dài."
)
result = tuned_pipeline(quick_check)
result

[{'entity': 'B-NAME',
  'score': np.float32(0.79037166),
  'index': 3,
  'word': 'Nguyễn',
  'start': None,
  'end': None},
 {'entity': 'I-NAME',
  'score': np.float32(0.70596457),
  'index': 4,
  'word': 'Văn',
  'start': None,
  'end': None},
 {'entity': 'I-NAME',
  'score': np.float32(0.7768921),
  'index': 5,
  'word': 'A@@',
  'start': None,
  'end': None},
 {'entity': 'B-AGE',
  'score': np.float32(0.7011042),
  'index': 7,
  'word': '35',
  'start': None,
  'end': None},
 {'entity': 'B-LOCATION',
  'score': np.float32(0.96342313),
  'index': 13,
  'word': 'Cầu',
  'start': None,
  'end': None},
 {'entity': 'I-LOCATION',
  'score': np.float32(0.97164977),
  'index': 14,
  'word': 'Giấ@@',
  'start': None,
  'end': None},
 {'entity': 'I-LOCATION',
  'score': np.float32(0.96303445),
  'index': 15,
  'word': 'y@@',
  'start': None,
  'end': None},
 {'entity': 'B-LOCATION',
  'score': np.float32(0.9654699),
  'index': 17,
  'word': 'Hà',
  'start': None,
  'end': None},
 {'entity': '

In [52]:
model_config = {"batch_size": 8}
signature = mlflow.models.infer_signature(
    [quick_check],
    mlflow.transformers.generate_signature_output(tuned_pipeline, [quick_check]),
    params=model_config,
)

In [53]:
with mlflow.start_run(run_id=run.info.run_id) :
    model_info = mlflow.transformers.log_model(
        transformers_model=tuned_pipeline,
        name="fine_tuned",
        signature=signature,
        input_example=[quick_check],
        model_config=model_config,
    )

2026/01/27 15:55:33 WARNING mlflow.utils.requirements_utils: Found torch version (2.9.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.9.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/01/27 15:55:33 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.24.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.24.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/01/27 15:55:45 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.24.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.24.0' without the local

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
2026/01/27 15:55:45 WARNING mlflow.transformers: params provided to the `predict` method will override the inference configuration saved with the model. If the params provided are not valid for the pipeline, MlflowException will be raised.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


🏃 View run whimsical-sheep-667 at: http://34.126.82.240:5000/#/experiments/1/runs/562ac232d754457996a75a0ce42358a6
🧪 View experiment at: http://34.126.82.240:5000/#/experiments/1
